# Check fangivingness

In [ ]:
import SimplicialComplex as sc
import sympy as sp
import numpy as np
from IPython.display import display

def no_sol_free_symbol(M,b): #M with 1 column and system of the form M x >= b
    for k in range(M.shape[0]):
        expr = M[k]
        if len((sp.simplify(expr)>=0).free_symbols)==0 and sp.simplify(expr)<b[k]:
            return True
    return False

def execute_free_symb_FM(M,b): # of the form Mx >=b
    filter_no_free_vars = np.zeros(M.shape,dtype=bool)
    for k in range(M.shape[0]):
        for l in range(M.shape[1]):
            filter_no_free_vars[k,l] = len((M[k,l]>0).free_symbols)==0
    if np.any(np.all(filter_no_free_vars,axis=0)): # find a column free of variables, and apply the Fourrier-Motzkin reduction w.r.t. this column
        # print(np.where(np.all(filter_no_free_vars,axis=0))[0])
        col_fm_index = np.where(np.all(filter_no_free_vars,axis=0))[0][0]
        col_fm = np.array(M[:,col_fm_index])
        pos = np.where(col_fm>0)[0]
        neg = np.where(col_fm<0)[0]
        zer = np.where(col_fm==0)[0]
        new_M = sp.zeros(len(pos)*len(neg)+len(zer),M.shape[1]-1)
        new_b = sp.zeros(len(pos)*len(neg)+len(zer),1)
        index_row = 0
        complement_col = [k for k in range(M.shape[1]) if k!=col_fm_index]
        for k in zer:
            new_M[index_row,:] = M[k,complement_col]
            b[index_row] = b[k]
            index_row+=1
        for k in pos:
            for l in neg:
                new_M[index_row,:] = M[k,complement_col]*abs(col_fm[l]) + M[l,complement_col]*col_fm[k]
                new_b[index_row] = b[k]*abs(col_fm[l]) + b[l]*col_fm[k]
                index_row+=1
        return(new_M,new_b)
    else:
        return False
    

def is_fangiving(K:sc.PureSimplicialComplex,char_map:sp.Matrix,variables):
    N = len(K.facets_bin)
    char_map_sp = sp.Matrix(char_map)
    for i in range(N): # For each pair of facets
        facet_bin1 = K.facets_bin[i]
        facet_set1 = sc.binary_to_face_0(facet_bin1,K.m)
        for j in range(i+1,N):
            facet_bin2 = K.facets_bin[j]
            inter_bin = facet_bin1 & facet_bin2
            inter_set= sc.binary_to_face_0(inter_bin,K.m)
            small_bin1 = facet_bin1 ^ inter_bin
            small_set1 = sc.binary_to_face_0(small_bin1,K.m)
            small_bin2 = facet_bin2 ^ inter_bin
            small_set2 = sc.binary_to_face_0(small_bin2,K.m)
            A = sp.zeros(K.n,len(facet_set1)+len(small_set2))
            A[:,:len(small_set1)] = char_map_sp[:,small_set1].copy()
            A[:,len(small_set1):len(small_set1)+len(inter_set)] = char_map_sp[:,inter_set].copy()
            A[:,len(facet_set1):] = char_map_sp[:,small_set2].copy()
            nullspace= [list(r) for r in A.nullspace()]
            if len(nullspace)==0: # nullspace of dim 0
                continue
            elif len(nullspace)==1: # nullspace of dim 1: just check the coefficients of the vector
                single_v = [nullspace[0][k] for k in range(len(small_set1))]
                test=True
                for expr in single_v:
                    if len((expr>=0).free_symbols)==0 and expr>0:
                        test = False
                        continue
                if test:    
                    print("There are overlapping fans")
            else: # nullspace of dim>1: need to perform successive Fourrier Motzkin reductions to be back to dim 1
                test = False
                D = sp.Array([v for v in nullspace]).transpose()
                b = sp.zeros(len(small_set1)+len(small_set2)+1,1)
                B = sp.zeros(len(small_set1)+len(small_set2)+1,len(nullspace))
                B[:len(small_set1),:] = -D[:len(small_set1),:]
                B[len(small_set1):-1,:] = D[len(facet_set1):,:]
                B[-1,:] = sp.ones(1,len(nullspace))
                b[-1]=1
                for k in range(B.shape[0]):
                    v = B[k,:]
                    test = True
                    for expr in v:
                        if len((expr>=0).free_symbols)==0:                        
                            test &= expr<0
                        else:
                            test = False
                    if test:
                        break
                if not test:
                    while B.shape[1]>1:
                        res = execute_free_symb_FM(B,b)
                        if not res:
                            break
                        else:
                            B,b = res
                    if B.shape[1]==1:
                        if no_sol_free_symbol(B,b):
                            continue
                        else:
                            display(B,b)
                    else:
                        for k in range(B.shape[0]):
                            v = B[k,:]
                            test = True
                            for expr in v:
                                if len((expr>=0).free_symbols)==0:                        
                                    test &= expr<0
                                else:
                                    test = False
                            if test:# means we are good
                                break
                        if not test:
                            filter_no_free_vars = np.zeros(B.shape,dtype=bool)
                            for k in range(B.shape[0]):
                                for l in range(B.shape[1]):
                                    filter_no_free_vars[k,l] = len((B[k,l]>0).free_symbols)==0
                            subsystem_indexes = np.where(np.all(filter_no_free_vars,axis=1))[0]
                            # print(subsystem_indexes)
                            sub_B = sp.zeros(len(subsystem_indexes),B.shape[1])
                            sub_b = sp.zeros(len(subsystem_indexes),1)
                            for k in range(len(subsystem_indexes)):
                                sub_B[k,:] = B[subsystem_indexes[k],:]
                                sub_b[k] = b[subsystem_indexes[k]]
                            while sub_B.shape[1]>1: # Execute the Fourrier-Motzkin reduction on the part without variables
                                res = execute_free_symb_FM(sub_B,sub_b)
                                if not res:
                                    break
                                else:
                                    sub_B,sub_b = res
                            if sub_B.shape[1]==1:
                                if no_sol_free_symbol(sub_B,sub_b):
                                    continue
                                else:
                                    indexes_var = np.where( filter_no_free_vars == False)[0]
                                    new_B = sp.zeros(len(sub_B)+len(indexes_var)*(len(indexes_var)-1)//2,B.shape[1])
                                    new_b = sp.zeros(len(sub_B)+len(indexes_var)*(len(indexes_var)-1)//2,1)
                                    for k in range(len(sub_B)):
                                        new_B[k]=sub_B[k]
                                        new_b[k]=sub_b[k]
                                    index_row = len(sub_B)
                                    for k in range(len(indexes_var)): # this is a cheatcode for when the sum of two rows yields an inequality without variables
                                        for l in range(k+1,len(indexes_var)):
                                            new_B[index_row,:]=B[indexes_var[k],:]+B[indexes_var[l],:]
                                            new_b[index_row]=b[indexes_var[k]]+b[indexes_var[l]]
                                            index_row+=1
                                    while new_B.shape[1]>1: #In that case, we are sure there are no free variables
                                        res = execute_free_symb_FM(new_B,new_b)
                                        if not res:
                                            break
                                        else:
                                            new_B,new_b = res
                                    if new_B.shape[1]==1:
                                        if no_sol_free_symbol(new_B,new_b):
                                            continue
                                        else:
                                            display(new_B,new_b)

## $K_2^1$

In [15]:
K_2_1 = sc.PureSimplicialComplex(None,[[2, 4], [3, 6], [3, 7], [4, 6], [4, 7], [5, 7], [1, 2, 5], [1, 2, 6], [1, 3, 5]],3)
a0,a1,a2 = sp.symbols('a0:3')
char_map_0 = sp.Matrix([[1, 0, 0, a0, -1, 0, 1], [0, 1, 0, -1, 0, 1, 2], [0, 0, 1, a1, -1, -1, -1]])
is_fangiving(K_2_1,char_map_0,[a0,a1])
char_map_1 = sp.Matrix([[1, 0, 0, 0, -1, 0, 1], [0, 1, 0, -1, -1, 0, 1], [0, 0, 1, a0, a0 - 1, -1, -1]])
is_fangiving(K_2_1,char_map_1,[a0])
char_map_2 = sp.Matrix([[1, 0, 0, 0, -1, 0, 1], [0, 1, 0, -1, a0 - 2, a0 - 1, a0], [0, 0, 1, 0, -1, -1, -1]])
is_fangiving(K_2_1,char_map_2,[a0])
char_map_3 = sp.Matrix([[1, 0, 0, 0, -1, -1, a0], [0, 1, 0, -1, -1, -1, a1], [0, 0, 1, 2, 1, 0, -1]])
is_fangiving(K_2_1,char_map_3,[a0,a1])
char_map_4 = sp.Matrix([[1, 0, 0, 0, -1, 0, 1], [0, 1, 0, -1, a0 - 2, a0 - 1, a0], [0, 0, 1, 0, -1, -1, -1]])
is_fangiving(K_2_1,char_map_4,[a0])
char_map_5 = sp.Matrix([[1, 0, 0, 0, -1, -1, 0], [0, 1, 0, -1, a0 - 1, 2*a0 - 1, a0], [0, 0, 1, 0, -1, -2, -1]])
is_fangiving(K_2_1,char_map_5,[a0])
char_map_6 = sp.Matrix([[1, 0, 0, 0, -1, -a0 - 1, 1], [0, 1, 0, -1, -2, -2*a0 - 1, 2], [0, 0, 1, 1, 1, a0, -1]])
is_fangiving(K_2_1,char_map_6,[a0])
char_map_7 = sp.Matrix([[1, 0, 0, 0, -1, a0 - 1, a0], [0, 1, 0, -1, -1, a1 - 1, a1], [0, 0, 1, 1, 0, -1, -1]])
is_fangiving(K_2_1,char_map_7,[a0,a1])
char_map_8 = sp.Matrix([[1, 0, 0, 0, -1, 0, 1], [0, 1, 0, -1, a0, 1, 2], [0, 0, 1, 1, -a0 - 1, -1, -1]])
is_fangiving(K_2_1,char_map_8,[a0])
char_map_9 = sp.Matrix([[1, 0, 0, 0, -1, -1, 0], [0, 1, 0, -1, -a0 - 2, -a0 - 1, 1], [0, 0, 1, 1, a0 + 1, a0, -1]])
is_fangiving(K_2_1,char_map_9,[a0])
char_map_10 = sp.Matrix([[1, 0, 0, 0, -1, -1, 0], [0, 1, 0, -1, -1, -1, 0], [0, 0, 1, a0 + 2, a0 + 1, a0, -1]])
is_fangiving(K_2_1,char_map_10,[a0])
char_map_11 = sp.Matrix([[1, 0, 0, 0, -1, 1, 1], [0, 1, 0, -1, -1, 1, 1], [0, 0, 1, 2, 1, -2, -1]])
is_fangiving(K_2_1,char_map_11,[])
char_map_12 = sp.Matrix([[1, 0, 0, 0, -1, -a0 - 1, 1], [0, 1, 0, -1, -1, -a0 - 1, 1], [0, 0, 1, 2, 1, a0, -1]])
is_fangiving(K_2_1,char_map_12,[a0])
char_map_13 = sp.Matrix([[1, 0, 0, 0, -1, 0, 1], [0, 1, 0, -1, -1, 0, 1], [0, 0, 1, a0, a0 - 1, -1, -1]])
is_fangiving(K_2_1,char_map_13,[a0])

## $K_2^2$

In [16]:
K_2_2 = sc.PureSimplicialComplex(None,[[1, 4], [1, 7], [2, 5], [2, 7], [3, 6], [3, 7], [4, 5, 6]],3)
a0,a1,a2 = sp.symbols('a0:3')

char_map_1 = sp.Matrix([[1, 0, 0, -1, 1, 0, 0], [0, 1, 0, 0, -1, a0, -1], [0, 0, 1, 1, 0, -1, 0]])
is_fangiving(K_2_2,char_map_1,[a0])

char_map_2 = sp.Matrix([[1, 0, 0, -1, -1, 0, -1], [0, 1, 0, 0, -1, -1, -1], [0, 0, 1, -1, 0, -1, -1]])
is_fangiving(K_2_2,char_map_2,[])

char_map_3 = sp.Matrix([[1, 0, 0, -1, 0, a0, -1], [0, 1, 0, 1, -1, 0, 0], [0, 0, 1, 0, 1, -1, 0]])
is_fangiving(K_2_2,char_map_3,[a0])

char_map_4 = sp.Matrix([[1, 0, 0, -1, 0, 1, 0], [0, 1, 0, a0, -1, 0, -1], [0, 0, 1, 0, 1, -1, 0]])
is_fangiving(K_2_2,char_map_4,[a0])
char_map_5 = sp.Matrix([[1, 0, 0, -1, a0, 0, a0 - 1], [0, 1, 0, 0, -1, 0, -1], [0, 0, 1, a2, a1, -1, a2 + a1 - 1]])
is_fangiving(K_2_2,char_map_5,[a0,a1,a2])

char_map_6 = sp.Matrix([[1, 0, 0, -1, 0, 0, -1], [0, 1, 0, a1, -1, 0, a1 - 1], [0, 0, 1, a2, a0, -1, a2 + a0 - 1]])
is_fangiving(K_2_2,char_map_6,[a0,a1,a2])

char_map_7 = sp.Matrix([[1, 0, 0, -1, a0, a1, a0 + a1 - 1], [0, 1, 0, 0, -1, 0, -1], [0, 0, 1, 0, a2, -1, a2 - 1]])
is_fangiving(K_2_2,char_map_7,[a0,a1,a2])

char_map_8 = sp.Matrix([[1, 0, 0, -1, a0, a1, a0 + a1 - 1], [0, 1, 0, 0, -1, a2, a2 - 1], [0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_2_2,char_map_8,[a0,a1,a2])

char_map_9 = sp.Matrix([[1, 0, 0, -1, 0, a0, a0 - 1], [0, 1, 0, a1, -1, a2, a1 + a2 - 1], [0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_2_2,char_map_9,[a0,a1,a2])

char_map_10 = sp.Matrix([[1, 0, 0, -1, 0, 0, -1], [0, 1, 0, a1, -1, a0, a1 + a0 - 1], [0, 0, 1, a2, 0, -1, a2 - 1]])
is_fangiving(K_2_2,char_map_10,[a0,a1,a2])

char_map_11 = sp.Matrix([[1, 0, 0, -1, 0, 1, 0], [0, 1, 0, 1, -1, 0, 0], [0, 0, 1, 0, a0, -1, -1]])
is_fangiving(K_2_2,char_map_11,[a0])

char_map_12 = sp.Matrix([[1, 0, 0, -1, 0, -1, -1], [0, 1, 0, -1, -1, 0, -1], [0, 0, 1, 0, -1, -1, -1]])
is_fangiving(K_2_2,char_map_12,[])

char_map_13 = sp.Matrix([[1, 0, 0, -1, a0, 0, -1], [0, 1, 0, 0, -1, 1, 0], [0, 0, 1, 1, 0, -1, 0]])
is_fangiving(K_2_2,char_map_13,[a0])

char_map_14 = sp.Matrix([[1, 0, 0, -1, 1, 0, 0], [0, 1, 0, 0, -1, 1, 0], [0, 0, 1, a0, 0, -1, -1]])
is_fangiving(K_2_2,char_map_14,[a0])

char_map_15 = sp.Matrix([[1, 0, 0, -1, 1, 0, a0], [0, 1, 0, 0, -1, 1, a1], [0, 0, 1, 1, 0, -1, -a0 - a1 - 1]])
is_fangiving(K_2_2,char_map_15,[a0,a1])

char_map_16 = sp.Matrix([[1, 0, 0, -1, 0, 1, a0], [0, 1, 0, 1, -1, 0, a1], [0, 0, 1, 0, 1, -1, -a0 - a1 - 1]])
is_fangiving(K_2_2,char_map_16,[a0,a1])

## $K_2^3$

In [17]:
K_2_3 = sc.PureSimplicialComplex(None,[[1, 7], [2, 4], [2, 5], [3, 5], [3, 6], [4, 6]],3)
a0,a1,a2,a3 = sp.symbols('a0:4')


char_map_1 = sp.Matrix([[1, 0, 0, 0, 0, 0, -1], [0, 1, 0, -1, -1, a0, a1], [0, 0, 1, 1, 0, -1, a2]])
char_map_2 = sp.Matrix([[1, 0, 0, a0, a1, a2, -1], [0, 1, 0, -1, -1, a3, 0], [0, 0, 1, 1, 0, -1, 0]])
char_map_3 = sp.Matrix([[1, 0, 0, 0, 0, a0, -1], [0, 1, 0, -1, -1, a1, a2], [0, 0, 1, 1, 0, -1, 0]])
char_map_4 = sp.Matrix([[1, 0, 0, 0, 0, 0, -1], [0, 1, 0, -1, -1, 0, a0], [0, 0, 1, a2, a2 - 1, -1, a1]])
char_map_5 = sp.Matrix([[1, 0, 0, a0, a1, a3, -1], [0, 1, 0, -1, -1, 0, 0], [0, 0, 1, a2, a2 - 1, -1, 0]])
char_map_6 = sp.Matrix([[1, 0, 0, a0, a0, 0, -1], [0, 1, 0, -1, -1, 0, 0], [0, 0, 1, a2, a2 - 1, -1, a1]])
char_map_7 = sp.Matrix([[1, 0, 0, 0, 0, 0, -1], [0, 1, 0, -1, a1, a1 + 1, a0], [0, 0, 1, 0, -1, -1, a2]])
char_map_8 = sp.Matrix([[1, 0, 0, a0, a1, a2, -1], [0, 1, 0, -1, a3, a3 + 1, 0], [0, 0, 1, 0, -1, -1, 0]])
char_map_9 = sp.Matrix([[1, 0, 0, 0, a0, a0, -1], [0, 1, 0, -1, a1, a1 + 1, a2], [0, 0, 1, 0, -1, -1, 0]])
char_map_10 = sp.Matrix([[1, 0, 0, 0, 0, 0, -1], [0, 1, 0, -1, 0, 1, a0], [0, 0, 1, a2, -1, -1, a1]])
char_map_11 = sp.Matrix([[1, 0, 0, a0, a1, a3, -1], [0, 1, 0, -1, 0, 1, 0], [0, 0, 1, a2, -1, -1, 0]])
char_map_12 = sp.Matrix([[1, 0, 0, a0, 0, 0, -1], [0, 1, 0, -1, 0, 1, 0], [0, 0, 1, a2, -1, -1, a1]])
char_map_13 = sp.Matrix([[1, 0, 0, 0, 0, 0, -1], [0, 1, 0, -1, a0, 1, a1], [0, 0, 1, 1, -a0 - 1, -1, a2]])
char_map_14 = sp.Matrix([[1, 0, 0, a0, a1, a2, -1], [0, 1, 0, -1, a3, 1, 0], [0, 0, 1, 1, -a3 - 1, -1, 0]])
char_map_15 = sp.Matrix([[1, 0, 0, 0, a0, 0, -1], [0, 1, 0, -1, a1, 1, a2], [0, 0, 1, 1, -a1 - 1, -1, -a2]])

is_fangiving(K_2_3,char_map_1,[a0,a1,a2])
is_fangiving(K_2_3,char_map_2,[a0,a1,a2,a3])
is_fangiving(K_2_3,char_map_3,[a0,a1,a2])
is_fangiving(K_2_3,char_map_4,[a0,a1,a2])
is_fangiving(K_2_3,char_map_5,[a0,a1,a2,a3])
is_fangiving(K_2_3,char_map_6,[a0,a1,a2])
is_fangiving(K_2_3,char_map_7,[a0,a1,a2])
is_fangiving(K_2_3,char_map_8,[a0,a1,a2,a3])
is_fangiving(K_2_3,char_map_9,[a0,a1,a2])
is_fangiving(K_2_3,char_map_10,[a0,a1,a2])
is_fangiving(K_2_3,char_map_11,[a0,a1,a2,a3])
is_fangiving(K_2_3,char_map_12,[a0,a1,a2])
is_fangiving(K_2_3,char_map_13,[a0,a1,a2])
is_fangiving(K_2_3,char_map_14,[a0,a1,a2,a3])
is_fangiving(K_2_3,char_map_15,[a0,a1,a2])

## $K_2^4$

In [18]:
K_2_4=sc.PureSimplicialComplex(None, [[2, 5], [3, 5], [3, 6], [3, 7], [4, 7], [5, 7], [1, 2, 4], [1, 2, 6], [1, 4, 6]] ,3)
d0=sp.symbols('d0')
d1=sp.symbols('d1')
c2=sp.symbols('c2')
b0=sp.symbols('b0')
b1=sp.symbols('b1')
char_map_0 =sp. Matrix([[1, 0, 0, -1, 0, 0, 1], [0, 1, 0, -1, 0, 1, 2], [0, 0, 1, a2, -1, -1, -1]])
is_fangiving(K_2_4,char_map_0 , [a2] )
char_map_1 =sp. Matrix([[1, 0, 0, -1, -1, -1, d0], [0, 1, 0, -1, -2, -1, d1], [0, 0, 1, 1, 1, 0, -1]])
is_fangiving(K_2_4,char_map_1 , [d0, d1] )
char_map_2 =sp. Matrix([[1, 0, 0, -1, d0 - 1, d0 - 1, d0], [0, 1, 0, -1, d1 - 2, d1 - 1, d1], [0, 0, 1, 0, -1, -1, -1]])
is_fangiving(K_2_4,char_map_2 , [d0, d1] )
char_map_3 =sp. Matrix([[1, 0, 0, -1, -1, -c2 - 1, 1], [0, 1, 0, -1, -2, -2*c2 - 1, 2], [0, 0, 1, 1, 1, c2, -1]])
is_fangiving(K_2_4,char_map_3 , [c2] )
char_map_4 =sp. Matrix([[1, 0, 0, -1, 0, 0, 1], [0, 1, 0, -1, -1, d1 - 1, d1], [0, 0, 1, 1, 0, -1, -1]])
is_fangiving(K_2_4,char_map_4 , [d1] )
char_map_5 =sp. Matrix([[1, 0, 0, -1, b0, 0, 1], [0, 1, 0, -1, b1, 1, 2], [0, 0, 1, 1, -b1 - 1, -1, -1]])
is_fangiving(K_2_4,char_map_5 , [b1, b0] )
char_map_6 =sp. Matrix([[1, 0, 0, -1, -c2 - 1, -c2 - 1, 1], [0, 1, 0, -1, -c2 - 2, -c2 - 1, 1], [0, 0, 1, 1, c2 + 1, c2, -1]])
is_fangiving(K_2_4,char_map_6 , [c2] )
char_map_7 =sp. Matrix([[1, 0, 0, -1, -1, -1, 0], [0, 1, 0, -1, -2, -1, 0], [0, 0, 1, c2 + 1, 2*c2 + 1, c2, -1]])
is_fangiving(K_2_4,char_map_7 , [c2] )
char_map_8 =sp. Matrix([[1, 0, 0, -1, 0, 0, 1], [0, 1, 0, -1, -1, 0, 1], [0, 0, 1, a2, a2 - 1, -1, -1]])
is_fangiving(K_2_4,char_map_8 , [a2] )

## $K_3^5$ -> OK

## $K_3^6$

In [ ]:

d1=sp.symbols('d1')
c3=sp.symbols('c3')
d3=sp.symbols('d3')
K_3_6=sc.PureSimplicialComplex(None, [[3, 7], [3, 8], [5, 7], [5, 8], [1, 2, 7], [1, 3, 6], [2, 4, 5], [4, 6, 8], [1, 2, 4, 6]] ,4)
char_map_0 =sp. Matrix([[1, 0, 0, 0, a0, -1, 0, 1], [0, 1, 0, 0, -1, 0, 1, 2], [0, 0, 1, 0, a2, -1, -1, -1], [0, 0, 0, 1, -1, 0, 1, 1]])
is_fangiving(K_3_6,char_map_0 , [a0, a2] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, a0, -1, -1, 0], [0, 1, 0, 0, -1, 0, 1, 1], [0, 0, 1, 0, a2, -1, -2, -1], [0, 0, 0, 1, -1, 0, 2, 1]])
is_fangiving(K_3_6,char_map_1 , [a0, a2] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, d0], [0, 1, 0, 0, -1, -2, -1, d1], [0, 0, 1, 0, 1, 1, 0, -1], [0, 0, 0, 1, -1, -2, 0, 1]])
is_fangiving(K_3_6,char_map_2 , [d1, d0] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, d0], [0, 1, 0, 0, -1, -1, -1, d1], [0, 0, 1, 0, 2, 1, 0, -1], [0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_3_6,char_map_3 , [d1, d0] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, -1, d1 - 2, d1 - 1, d1], [0, 0, 1, 0, 0, -1, -1, -1], [0, 0, 0, 1, -1, d3 - 1, d3, d3]])
is_fangiving(K_3_6,char_map_4 , [d1, d3] )
char_map_5 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, -1, d1 - 1, 2*d1 - 1, d1], [0, 0, 1, 0, 0, -1, -2, -1], [0, 0, 0, 1, -1, d3 - 1, 2*d3, d3]])
is_fangiving(K_3_6,char_map_5 , [d1, d3] )
char_map_6 =sp. Matrix([[1, 0, 0, 0, 0, -1, -c2 - 1, 1], [0, 1, 0, 0, -1, -2, -2*c2 - 1, 2], [0, 0, 1, 0, 1, 1, c2, -1], [0, 0, 0, 1, -1, -2, c3, 1]])
is_fangiving(K_3_6,char_map_6 , [c3, c2] )
char_map_7 =sp. Matrix([[1, 0, 0, 0, 0, -1, d0 - 1, d0], [0, 1, 0, 0, -1, -1, d1 - 1, d1], [0, 0, 1, 0, 1, 0, -1, -1], [0, 0, 0, 1, -1, -1, d3, d3]])
is_fangiving(K_3_6,char_map_7 , [d1, d0, d3] )
char_map_8 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, -1, b1, 1, 2], [0, 0, 1, 0, 1, -b1 - 1, -1, -1], [0, 0, 0, 1, -1, b1, 1, 1]])
is_fangiving(K_3_6,char_map_8 , [b1] )
char_map_9 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, -1, -c2 - 2, -c2 - 1, 1], [0, 0, 1, 0, 1, c2 + 1, c2, -1], [0, 0, 0, 1, -1, -c2 - 2, -c2, 1]])
is_fangiving(K_3_6,char_map_9 , [c2] )
char_map_10 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, -1, -1, -1, 0], [0, 0, 1, 0, c2 + 2, c2 + 1, c2, -1], [0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_3_6,char_map_10 , [c2] )
char_map_11 =sp. Matrix([[1, 0, 0, 0, 0, -1, -c2 - 1, 1], [0, 1, 0, 0, -1, -1, -c2 - 1, 1], [0, 0, 1, 0, 2, 1, c2, -1], [0, 0, 0, 1, -1, -1, c3, 0]])
is_fangiving(K_3_6,char_map_11 , [c3, c2] )
char_map_12 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, a2, a2 - 1, -1, -1], [0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_3_6,char_map_12 , [a2] )

## $K_3^7$

In [ ]:
b3=sp.symbols('b3')
K_3_7=sc.PureSimplicialComplex(None, [[2, 6], [3, 6], [3, 7], [6, 8], [1, 2, 5], [1, 2, 7], [3, 4, 8], [4, 5, 8], [1, 4, 5, 7]] ,4)
char_map_0 =sp. Matrix([[1, 0, 0, 0, -1, -1, -1, d0], [0, 1, 0, 0, -1, -2, -1, d1], [0, 0, 1, 0, 1, 1, 0, -1], [0, 0, 0, 1, 0, 1, 0, -1]])
is_fangiving(K_3_7,char_map_0 , [d1, d0] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, -1, d0 - 1, d0 - 1, d0], [0, 1, 0, 0, -1, d1 - 2, d1 - 1, d1], [0, 0, 1, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 1, 0, -1]])
is_fangiving(K_3_7,char_map_1 , [d0, d1] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, -1, -1, -c3 - 1, 1], [0, 1, 0, 0, -1, -2, -2*c3 - 1, 2], [0, 0, 1, 0, 1, 1, c3, -1], [0, 0, 0, 1, 0, 1, c3, -1]])
is_fangiving(K_3_7,char_map_2 , [c3] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, -1, -1, d1 - 1, d1], [0, 0, 1, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_3_7,char_map_3 , [d1] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, -1, b0, 0, 1], [0, 1, 0, 0, -1, b1, 1, 2], [0, 0, 1, 0, 1, -b1 - 1, -1, -1], [0, 0, 0, 1, 0, b3, -1, -1]])
is_fangiving(K_3_7,char_map_4 , [b1, b3, b0] )
char_map_5 =sp. Matrix([[1, 0, 0, 0, -1, -c2 - 1, -c2 - 1, 1], [0, 1, 0, 0, -1, -c2 - 2, -c2 - 1, 1], [0, 0, 1, 0, 1, c2 + 1, c2, -1], [0, 0, 0, 1, 0, c3 + 1, c3, -1]])
is_fangiving(K_3_7,char_map_5 , [c2, c3] )
char_map_6 =sp. Matrix([[1, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, -1, -2, -1, 0], [0, 0, 1, 0, c2 + 1, 2*c2 + 1, c2, -1], [0, 0, 0, 1, c3, 2*c3 + 1, c3, -1]])
is_fangiving(K_3_7,char_map_6 , [c2, c3] )
char_map_7 =sp. Matrix([[1, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, a2, a2 - 1, -1, -1], [0, 0, 0, 1, 0, 1, 0, -1]])
is_fangiving(K_3_7,char_map_7 , [a2] )

## $K_3^8$

In [30]:
d3=sp.symbols('d3')
c0=sp.symbols('c0')
b2 = sp.symbols('b2')
K_3_8=sc.PureSimplicialComplex(None, [[2, 6], [2, 8], [3, 7], [1, 4, 5], [1, 4, 8], [1, 5, 6], [3, 4, 8], [5, 6, 7]] ,4)
char_map_0 =sp. Matrix([[1, 0, 0, 0, -1, -1, 0, -1], [0, 1, 0, 0, 0, -1, -1, -1], [0, 0, 1, 0, -1, 0, -1, -1], [0, 0, 0, 1, -1, 0, 0, -1]])
is_fangiving(K_3_8,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, -1, 0, c0, -1], [0, 1, 0, 0, 1, -1, 0, 0], [0, 0, 1, 0, 0, 1, -1, 0], [0, 0, 0, 1, -1, 1, c3, -1]])
is_fangiving(K_3_8,char_map_1 , [c3, c0] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, -1, 0, 1, 0], [0, 1, 0, 0, a1, -1, 0, -1], [0, 0, 1, 0, 0, 1, -1, 0], [0, 0, 0, 1, -1, 1, 0, 0]])
is_fangiving(K_3_8,char_map_2 , [a1] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, -1, 0, 0, -1], [0, 1, 0, 0, 1, -1, 0, 0], [0, 0, 1, 0, a2, b2, -1, a2 + b2 - 1], [0, 0, 0, 1, -1, 1, 0, -1]])
is_fangiving(K_3_8,char_map_3 , [a2, b2] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, -1, b0, 1, b0], [0, 1, 0, 0, 0, -1, 0, -1], [0, 0, 1, 0, 0, b2, -1, b2 - 1], [0, 0, 0, 1, -1, b3, 0, b3 - 1]])
is_fangiving(K_3_8,char_map_4 , [b0, b2, b3] )
char_map_5 =sp. Matrix([[1, 0, 0, 0, -1, 0, c0, c0 - 1], [0, 1, 0, 0, 1, -1, -d3 - 1, -d3 - 1], [0, 0, 1, 0, 0, 0, -1, -1], [0, 0, 0, 1, -1, 1, d3 + 1, d3]])
is_fangiving(K_3_8,char_map_5 , [d3, c0] )
char_map_6 =sp. Matrix([[1, 0, 0, 0, -1, 0, 1, 0], [0, 1, 0, 0, a1, -1, 0, a1 - 1], [0, 0, 1, 0, 0, 0, -1, -1], [0, 0, 0, 1, -1, 0, 0, -1]])
is_fangiving(K_3_8,char_map_6 , [a1] )
char_map_7 =sp. Matrix([[1, 0, 0, 0, -1, 0, 0, -1], [0, 1, 0, 0, 1, -1, -1, -1], [0, 0, 1, 0, 1, 0, -1, 0], [0, 0, 0, 1, -1, 1, 0, 0]])
is_fangiving(K_3_8,char_map_7 , [] )
char_map_8 =sp. Matrix([[1, 0, 0, 0, -1, 0, 1, 0], [0, 1, 0, 0, 1, -1, 0, 0], [0, 0, 1, 0, 0, b2, -1, -1], [0, 0, 0, 1, -1, b3, 0, -1]])
is_fangiving(K_3_8,char_map_8 , [b2, b3] )
char_map_9 =sp. Matrix([[1, 0, 0, 0, -1, 0, 1, d0], [0, 1, 0, 0, 1, -1, 0, d1], [0, 0, 1, 0, 0, 1, -1, -d0 - d1 - 1], [0, 0, 0, 1, -1, 1, 0, -d1 - 1]])
is_fangiving(K_3_8,char_map_9 , [d1, d0] )

## $K_3^9$

In [31]:
K_3_9=sc.PureSimplicialComplex(None, [[2, 5], [5, 8], [6, 8], [1, 2, 6], [1, 2, 7], [1, 3, 6], [3, 4, 7], [3, 4, 8], [4, 5, 7]] ,4)
char_map_0 =sp. Matrix([[1, 0, 0, 0, a0, -1, 0, 1], [0, 1, 0, 0, -1, 0, 1, 2], [0, 0, 1, 0, a2, -1, -1, -1], [0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_3_9,char_map_0 , [a0, a2] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 1, -1, -1, 0], [0, 1, 0, 0, -1, 0, 1, 1], [0, 0, 1, 0, 2, -1, -2, -1], [0, 0, 0, 1, 1, -1, -2, -1]])
is_fangiving(K_3_9,char_map_1 , [] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, -1, -2, -1, 0], [0, 0, 1, 0, 1, 1, 0, -1], [0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_3_9,char_map_2 , [] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, d0], [0, 1, 0, 0, -1, -1, -1, d1], [0, 0, 1, 0, 2, 1, 0, -1], [0, 0, 0, 1, 1, 1, 0, -1]])
is_fangiving(K_3_9,char_map_3 , [d0, d1] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, -1, d1 - 2, d1 - 1, d1], [0, 0, 1, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_3_9,char_map_4 , [d1] )
char_map_5 =sp. Matrix([[1, 0, 0, 0, 0, -1, d0 - 1, d0], [0, 1, 0, 0, -1, -1, d1 - 1, d1], [0, 0, 1, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_3_9,char_map_5 , [d0, d1] )
char_map_6 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, -1, b1, 1, 2], [0, 0, 1, 0, 1, -b1 - 1, -1, -1], [0, 0, 0, 1, 0, b3, -1, -1]])
is_fangiving(K_3_9,char_map_6 , [b3, b1] )
char_map_7 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, -1, -1, -1, 0], [0, 0, 1, 0, c2 + 2, c2 + 1, c2, -1], [0, 0, 0, 1, c3 + 1, c3 + 1, c3, -1]])
is_fangiving(K_3_9,char_map_7 , [c2, c3] )
char_map_8 =sp. Matrix([[1, 0, 0, 0, 0, -1, -c2 - 1, 1], [0, 1, 0, 0, -1, -1, -c2 - 1, 1], [0, 0, 1, 0, 2, 1, c2, -1], [0, 0, 0, 1, 1, 1, c2, -1]])
is_fangiving(K_3_9,char_map_8 , [c2] )
char_map_9 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, a2, a2 - 1, -1, -1], [0, 0, 0, 1, a3, a3, -1, -1]])
is_fangiving(K_3_9,char_map_9 , [a3, a2] )

## $K_3^{10}$

In [33]:
K_3_10=sc.PureSimplicialComplex(None, [[3, 7], [6, 8], [1, 2, 5], [1, 2, 6], [1, 2, 7], [1, 3, 6], [2, 4, 5], [3, 4, 8], [4, 5, 7], [4, 5, 8]] ,4)
char_map_0 =sp. Matrix([[1, 0, 0, 0, -1, -1, 0, 1], [0, 1, 0, 0, -1, 0, 1, 2], [0, 0, 1, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 1, 0, -1]])
is_fangiving(K_3_10,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, -1, -2, -1, 0], [0, 0, 1, 0, 0, -1, -1, -1], [0, 0, 0, 1, -1, -1, -1, -1]])
is_fangiving(K_3_10,char_map_1 , [] )

## $K_3^{11}$

In [35]:
K_3_11=sc.PureSimplicialComplex(None, [[2, 6], [2, 8], [1, 2, 5], [1, 4, 5], [1, 4, 8], [1, 5, 7], [3, 4, 7], [3, 4, 8], [3, 6, 7], [3, 6, 8], [5, 6, 7]] ,4)
char_map_0 =sp. Matrix([[1, 0, 0, 0, -1, 1, 0, 0], [0, 1, 0, 0, 0, -1, 2, 1], [0, 0, 1, 0, 0, 0, -1, -1], [0, 0, 0, 1, -1, 1, -1, -1]])
is_fangiving(K_3_11,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, -1, 0, 0, -1], [0, 1, 0, 0, -2, -1, 2, -1], [0, 0, 1, 0, 1, 0, -1, 0], [0, 0, 0, 1, 1, 1, -1, 0]])
is_fangiving(K_3_11,char_map_1 , [] )

## $K_3^{12}$

In [36]:
K_3_12=sc.PureSimplicialComplex(None, [[2, 5], [6, 8], [1, 2, 6], [1, 2, 7], [1, 3, 6], [1, 3, 7], [3, 4, 7], [3, 4, 8], [4, 5, 7], [4, 5, 8]] ,4)
char_map_0 =sp. Matrix([[1, 0, 0, 0, a0, -1, -1, 0], [0, 1, 0, 0, -1, 0, 1, 1], [0, 0, 1, 0, a2, -1, -2, -1], [0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_3_12,char_map_0 , [a2, a0] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, d0], [0, 1, 0, 0, -1, -2, -1, d1], [0, 0, 1, 0, 1, 1, 0, -1], [0, 0, 0, 1, 0, 1, 0, -1]])
is_fangiving(K_3_12,char_map_1 , [d1, d0] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, -1, d1 - 1, 2*d1 - 1, d1], [0, 0, 1, 0, 0, -1, -2, -1], [0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_3_12,char_map_2 , [d1] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, -1, -c3 - 1, 1], [0, 1, 0, 0, -1, -2, -2*c3 - 1, 2], [0, 0, 1, 0, 1, 1, c3, -1], [0, 0, 0, 1, 0, 1, c3, -1]])
is_fangiving(K_3_12,char_map_3 , [c3] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, 0, -1, d0 - 1, d0], [0, 1, 0, 0, -1, -1, d1 - 1, d1], [0, 0, 1, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_3_12,char_map_4 , [d1, d0] )
char_map_5 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, -1, b1, 1, 2], [0, 0, 1, 0, 1, -b1 - 1, -1, -1], [0, 0, 0, 1, 0, b3, -1, -1]])
is_fangiving(K_3_12,char_map_5 , [b3, b1] )
char_map_6 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, -1, -c2 - 2, -c2 - 1, 1], [0, 0, 1, 0, 1, c2 + 1, c2, -1], [0, 0, 0, 1, 0, c3 + 1, c3, -1]])
is_fangiving(K_3_12,char_map_6 , [c3, c2] )
char_map_7 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, -1, -1, -1, 0], [0, 0, 1, 0, c2 + 2, c2 + 1, c2, -1], [0, 0, 0, 1, c3 + 1, c3 + 1, c3, -1]])
is_fangiving(K_3_12,char_map_7 , [c3, c2] )

## $K_3^{13}$

In [37]:
K_3_13=sc.PureSimplicialComplex(None, [[2, 5], [3, 8], [5, 7], [5, 8], [6, 8], [1, 2, 6], [3, 4, 7], [1, 2, 4, 7], [1, 3, 4, 6]] ,4)
char_map_0 =sp. Matrix([[1, 0, 0, 0, a0, -1, 0, 1], [0, 1, 0, 0, -1, 0, 1, 2], [0, 0, 1, 0, a2, -1, -1, -1], [0, 0, 0, 1, a3, -1, -1, 0]])
is_fangiving(K_3_13,char_map_0 , [a2, a3, a0] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, d0], [0, 1, 0, 0, -1, -1, -1, d1], [0, 0, 1, 0, 2, 1, 0, -1], [0, 0, 0, 1, 1, 0, -1, d3]])
is_fangiving(K_3_13,char_map_1 , [d3, d0, d1] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, -1, d1 - 2, d1 - 1, d1], [0, 0, 1, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, -1, -1, 0]])
is_fangiving(K_3_13,char_map_2 , [d1] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, -1, d0 - 1, d0], [0, 1, 0, 0, -1, -1, d1 - 1, d1], [0, 0, 1, 0, 1, 0, -1, -1], [0, 0, 0, 1, 1, 0, -1, 0]])
is_fangiving(K_3_13,char_map_3 , [d0, d1] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, -1, b1, 1, 2], [0, 0, 1, 0, 1, -b1 - 1, -1, -1], [0, 0, 0, 1, 1, -b1 - 1, -1, 0]])
is_fangiving(K_3_13,char_map_4 , [b1] )
char_map_5 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, -1, -1, -1, 0], [0, 0, 1, 0, c2 + 2, c2 + 1, c2, -1], [0, 0, 0, 1, 1, 0, -1, 0]])
is_fangiving(K_3_13,char_map_5 , [c2] )
char_map_6 =sp. Matrix([[1, 0, 0, 0, 0, -1, -c2 - 1, 1], [0, 1, 0, 0, -1, -1, -c2 - 1, 1], [0, 0, 1, 0, 2, 1, c2, -1], [0, 0, 0, 1, 1, 0, -1, 0]])
is_fangiving(K_3_13,char_map_6 , [c2] )
char_map_7 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, a2, a2 - 1, -1, -1], [0, 0, 0, 1, a3, a3 - 1, -1, 0]])
is_fangiving(K_3_13,char_map_7 , [a2, a3] )

## $K_3^{14}$

In [39]:
K_3_14=sc.PureSimplicialComplex(None, [[2, 6], [5, 8], [1, 2, 5], [1, 2, 7], [1, 3, 7], [1, 5, 7], [3, 4, 6], [3, 4, 7], [3, 4, 8], [4, 6, 8]] ,4)
char_map_0 =sp. Matrix([[1, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, -1, -1, -1], [0, 0, 0, 1, 1, 0, 0, -1]])
is_fangiving(K_3_14,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 2, 1, -1, -1], [0, 0, 0, 1, 1, 0, -1, -1]])
is_fangiving(K_3_14,char_map_1 , [] )

## $K_4^{15}$

In [41]:
d4=sp.symbols('d4')
K_4_15=sc.PureSimplicialComplex(None, [[2, 7], [1, 4, 6], [1, 4, 9], [1, 6, 7], [2, 5, 9], [3, 5, 8], [3, 6, 8], [6, 7, 8], [3, 4, 5, 9]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 0, -1], [0, 1, 0, 0, 0, 0, -1, -1, -1], [0, 0, 1, 0, 0, -1, 0, -1, -1], [0, 0, 0, 1, 0, -1, 0, 0, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_4_15,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, c0, -1], [0, 1, 0, 0, 0, 1, -1, 0, 0], [0, 0, 1, 0, 0, 0, 1, -1, 0], [0, 0, 0, 1, 0, -1, 1, c3, -1], [0, 0, 0, 0, 1, 1, 0, -1, 0]])
is_fangiving(K_4_15,char_map_1 , [c3, c0] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 0, -1], [0, 1, 0, 0, 0, 1, -1, 0, 0], [0, 0, 1, 0, 0, a2, 1, -1, a2], [0, 0, 0, 1, 0, -1, 1, 0, -1], [0, 0, 0, 0, 1, d4 + 1, 0, -1, d4]])
is_fangiving(K_4_15,char_map_2 , [a2, d4] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 1, 0], [0, 1, 0, 0, 0, 0, -1, 0, -1], [0, 0, 1, 0, 0, 0, 1, -1, 0], [0, 0, 0, 1, 0, -1, 0, 0, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_4_15,char_map_3 , [] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 1, -1], [0, 1, 0, 0, 0, 0, -1, 0, -1], [0, 0, 1, 0, 0, 0, 1, -1, 0], [0, 0, 0, 1, 0, -1, 0, 0, -1], [0, 0, 0, 0, 1, 1, 1, -1, 0]])
is_fangiving(K_4_15,char_map_4 , [] )
char_map_5 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 1, 0], [0, 1, 0, 0, 0, 1, -1, -1, -1], [0, 0, 1, 0, 0, 0, 0, -1, -1], [0, 0, 0, 1, 0, -1, 1, 1, 0], [0, 0, 0, 0, 1, 1, 0, -1, -1]])
is_fangiving(K_4_15,char_map_5 , [] )

## $K_4^{16}$

In [42]:
b4=sp.symbols('b4')
K_4_16=sc.PureSimplicialComplex(None, [[2, 7], [7, 9], [1, 2, 6], [3, 4, 9], [3, 5, 7], [3, 5, 8], [4, 6, 9], [1, 2, 5, 8], [1, 4, 6, 8]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, -1, d0], [0, 1, 0, 0, 0, -1, -2, -1, d1], [0, 0, 1, 0, 0, 1, 1, 0, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 0, -1, -1, d4]])
is_fangiving(K_4_16,char_map_0 , [d1, d0, d4] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, -1, d0 - 1, d0 - 1, d0], [0, 1, 0, 0, 0, -1, d1 - 2, d1 - 1, d1], [0, 0, 1, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 0, -1, -1, 0]])
is_fangiving(K_4_16,char_map_1 , [d1, d0] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, -1, 0, 1, 2], [0, 0, 1, 0, 0, 1, -1, -1, -1], [0, 0, 0, 1, 0, 0, 0, -1, -1], [0, 0, 0, 0, 1, 0, -1, -1, 0]])
is_fangiving(K_4_16,char_map_2 , [] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, -1, -c2 - 1, -c2 - 1, 1], [0, 1, 0, 0, 0, -1, -c2 - 2, -c2 - 1, 1], [0, 0, 1, 0, 0, 1, c2 + 1, c2, -1], [0, 0, 0, 1, 0, 0, c3 + 1, c3, -1], [0, 0, 0, 0, 1, 0, -1, -1, 0]])
is_fangiving(K_4_16,char_map_3 , [c2, c3] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, -1, -2, -1, 0], [0, 0, 1, 0, 0, c2 + 1, 2*c2 + 1, c2, -1], [0, 0, 0, 1, 0, c3, 2*c3 + 1, c3, -1], [0, 0, 0, 0, 1, 0, -1, -1, 0]])
is_fangiving(K_4_16,char_map_4 , [c2, c3] )
char_map_5 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, a2, a2 - 1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, b4 + 1, b4, -1, 0]])
is_fangiving(K_4_16,char_map_5 , [b4, a2] )

## $K_4^{17}$

In [43]:
K_4_17=sc.PureSimplicialComplex(None, [[2, 7], [1, 6, 8], [2, 5, 9], [3, 4, 8], [3, 4, 9], [3, 7, 8], [3, 7, 9], [6, 7, 8], [1, 2, 5, 6], [1, 4, 5, 6], [1, 4, 5, 9]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, -1, 1, 0, 0], [0, 1, 0, 0, 0, 0, -1, 2, 1], [0, 0, 1, 0, 0, 0, 0, -1, -1], [0, 0, 0, 1, 0, -1, 1, -1, -1], [0, 0, 0, 0, 1, -1, 0, 1, 0]])
is_fangiving(K_4_17,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 0, -1], [0, 1, 0, 0, 0, -2, -1, 2, -1], [0, 0, 1, 0, 0, 1, 0, -1, 0], [0, 0, 0, 1, 0, 1, 1, -1, 0], [0, 0, 0, 0, 1, -1, 0, 1, -1]])
is_fangiving(K_4_17,char_map_1 , [] )

## $K_4^{18}$

In [44]:
K_4_18=sc.PureSimplicialComplex(None, [[2, 7], [1, 2, 6], [1, 6, 8], [2, 5, 9], [3, 4, 8], [3, 7, 8], [3, 7, 9], [6, 7, 8], [1, 4, 5, 6], [1, 4, 5, 9], [3, 4, 5, 9]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, -1, 1, 0, 0], [0, 1, 0, 0, 0, 0, -1, 2, 1], [0, 0, 1, 0, 0, 0, 0, -1, -1], [0, 0, 0, 1, 0, -1, 1, -1, -1], [0, 0, 0, 0, 1, -1, 0, 0, -1]])
is_fangiving(K_4_18,char_map_0 , [] )

## $K_4^{19}$

In [45]:
a4=sp.symbols('a4')
c4=sp.symbols('c4')
K_4_19=sc.PureSimplicialComplex(None, [[2, 7], [3, 7], [7, 9], [1, 2, 6], [1, 2, 8], [3, 5, 8], [4, 6, 9], [3, 4, 5, 9], [1, 4, 5, 6, 8]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, -1, d0], [0, 1, 0, 0, 0, -1, -2, -1, d1], [0, 0, 1, 0, 0, 1, 1, 0, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 1, 2, 0, -1]])
is_fangiving(K_4_19,char_map_0 , [d0, d1] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, -1, d0 - 1, d0 - 1, d0], [0, 1, 0, 0, 0, -1, d1 - 2, d1 - 1, d1], [0, 0, 1, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_4_19,char_map_1 , [d0, d1] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, -1, -c2 - 1, -c2 - 1, 1], [0, 1, 0, 0, 0, -1, -c2 - 2, -c2 - 1, 1], [0, 0, 1, 0, 0, 1, c2 + 1, c2, -1], [0, 0, 0, 1, 0, 0, c3 + 1, c3, -1], [0, 0, 0, 0, 1, 1, c2 + 2, c2, -1]])
is_fangiving(K_4_19,char_map_2 , [c2, c3] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, -1, -2, -1, 0], [0, 0, 1, 0, 0, c2 + 1, 2*c2 + 1, c2, -1], [0, 0, 0, 1, 0, c3, 2*c3 + 1, c3, -1], [0, 0, 0, 0, 1, c4 + 1, 2*c4 + 2, c4, -1]])
is_fangiving(K_4_19,char_map_3 , [c4, c3, c2] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, a2, a2 - 1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, a4, a4, -1, -1]])
is_fangiving(K_4_19,char_map_4 , [a2, a4] )

## $K_4^{20}$

In [46]:
K_4_20=sc.PureSimplicialComplex(None, [[3, 9], [6, 8], [2, 5, 6], [3, 4, 8], [5, 6, 9], [5, 7, 9], [1, 2, 4, 7], [1, 2, 4, 8], [1, 2, 5, 7], [1, 3, 4, 7]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 1, 0, 0, 0, -1, 0, 1, 2], [0, 0, 1, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 1, -1, 0, 1, 1]])
is_fangiving(K_4_20,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, 0, -1, -1, -1, 1], [0, 0, 1, 0, 0, 2, 1, 0, -1], [0, 0, 0, 1, 0, 1, 0, -1, 0], [0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_4_20,char_map_1 , [] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 1, 0, -1, 0], [0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_4_20,char_map_2 , [] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 1, -1, 0, 1, 0]])
is_fangiving(K_4_20,char_map_3 , [] )

## $K_4^{21}$

In [47]:
K_4_21=sc.PureSimplicialComplex(None, [[3, 8], [7, 9], [1, 2, 7], [1, 3, 7], [3, 4, 9], [4, 6, 9], [1, 2, 5, 6], [1, 2, 5, 8], [2, 4, 5, 6], [4, 5, 6, 8]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 1, 0, 0, 0, -1, 0, 1, 2], [0, 0, 1, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, -1, 0, 0, 1]])
is_fangiving(K_4_21,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, 0, -1, -2, -1, 0], [0, 0, 1, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, -1, -1, -1, -1], [0, 0, 0, 0, 1, -1, -1, -1, 0]])
is_fangiving(K_4_21,char_map_1 , [] )

## $K_4^{22}$ - >TODO things missing

## $K_4^{23}$

In [49]:
K_4_23=sc.PureSimplicialComplex(None, [[2, 7], [3, 7], [3, 8], [1, 2, 8], [3, 4, 9], [5, 7, 9], [1, 2, 5, 6], [1, 4, 6, 8], [4, 5, 6, 9]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, -1, d0], [0, 1, 0, 0, 0, -1, -2, -1, d1], [0, 0, 1, 0, 0, 1, 1, 0, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_4_23,char_map_0 , [d0, d1] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, -1, d0 - 1, d0 - 1, d0], [0, 1, 0, 0, 0, -1, d1 - 2, d1 - 1, d1], [0, 0, 1, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, -1, d4 - 1, d4, d4]])
is_fangiving(K_4_23,char_map_1 , [d0, d4, d1] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, -c3 - 1, 1], [0, 1, 0, 0, 0, -1, -2, -2*c3 - 1, 2], [0, 0, 1, 0, 0, 1, 1, c3, -1], [0, 0, 0, 1, 0, 0, 1, c3, -1], [0, 0, 0, 0, 1, -1, -1, c4, 0]])
is_fangiving(K_4_23,char_map_2 , [c3, c4] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, -1, -1, d1 - 1, d1], [0, 0, 1, 0, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, 0, -1, -1], [0, 0, 0, 0, 1, -1, 0, 1, 0]])
is_fangiving(K_4_23,char_map_3 , [d1] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, 0, -1, b0, 0, 1], [0, 1, 0, 0, 0, -1, b1, 1, 2], [0, 0, 1, 0, 0, 1, -b1 - 1, -1, -1], [0, 0, 0, 1, 0, 0, b3, -1, -1], [0, 0, 0, 0, 1, -1, b1 + 1, 1, 0]])
is_fangiving(K_4_23,char_map_4 , [b1, b0, b3] )
char_map_5 =sp. Matrix([[1, 0, 0, 0, 0, -1, -c2 - 1, -c2 - 1, 1], [0, 1, 0, 0, 0, -1, -c2 - 2, -c2 - 1, 1], [0, 0, 1, 0, 0, 1, c2 + 1, c2, -1], [0, 0, 0, 1, 0, 0, c3 + 1, c3, -1], [0, 0, 0, 0, 1, -1, c4 - 1, c4, 0]])
is_fangiving(K_4_23,char_map_5 , [c2, c3, c4] )
char_map_6 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, -1, -2, -1, 0], [0, 0, 1, 0, 0, c2 + 1, 2*c2 + 1, c2, -1], [0, 0, 0, 1, 0, c3, 2*c3 + 1, c3, -1], [0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_4_23,char_map_6 , [c2, c3] )
char_map_7 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, a2, a2 - 1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_4_23,char_map_7 , [a2] )

## $K_4^{24}$

In [50]:
K_4_24=sc.PureSimplicialComplex(None, [[1, 2, 7], [1, 2, 8], [1, 3, 7], [1, 3, 8], [2, 5, 6], [3, 4, 8], [3, 4, 9], [3, 7, 9], [4, 6, 8], [5, 7, 9], [4, 5, 6, 9]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, d0], [0, 1, 0, 0, 0, -1, -2, -1, d1], [0, 0, 1, 0, 0, 1, 1, 0, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_4_24,char_map_0 , [d0, d1] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -c3 - 1, 1], [0, 1, 0, 0, 0, -1, -2, -2*c3 - 1, 2], [0, 0, 1, 0, 0, 1, 1, c3, -1], [0, 0, 0, 1, 0, 0, 1, c3, -1], [0, 0, 0, 0, 1, -1, -1, c4, 0]])
is_fangiving(K_4_24,char_map_1 , [c3, c4] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, d0 - 1, d0], [0, 1, 0, 0, 0, -1, -1, d1 - 1, d1], [0, 0, 1, 0, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, 0, -1, -1], [0, 0, 0, 0, 1, -1, 0, 1, 0]])
is_fangiving(K_4_24,char_map_2 , [d0, d1] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, 0, -1, b1, 1, 2], [0, 0, 1, 0, 0, 1, -b1 - 1, -1, -1], [0, 0, 0, 1, 0, 0, b3, -1, -1], [0, 0, 0, 0, 1, -1, b1 + 1, 1, 0]])
is_fangiving(K_4_24,char_map_3 , [b1, b3] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, 0, -1, -c2 - 2, -c2 - 1, 1], [0, 0, 1, 0, 0, 1, c2 + 1, c2, -1], [0, 0, 0, 1, 0, 0, c3 + 1, c3, -1], [0, 0, 0, 0, 1, -1, c4 - 1, c4, 0]])
is_fangiving(K_4_24,char_map_4 , [c3, c4, c2] )

## $K_4^{25}$

In [51]:
K_4_25=sc.PureSimplicialComplex(None, [[7, 9], [1, 2, 6], [2, 5, 7], [3, 4, 8], [3, 4, 9], [3, 5, 7], [3, 5, 8], [4, 6, 9], [1, 2, 5, 8], [1, 4, 6, 8]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, -1, d0], [0, 1, 0, 0, 0, -1, -2, -1, d1], [0, 0, 1, 0, 0, 1, 1, 0, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 0, -1, -1, d4]])
is_fangiving(K_4_25,char_map_0 , [d1, d0, d4] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, -c3 - 1, 1], [0, 1, 0, 0, 0, -1, -2, -2*c3 - 1, 2], [0, 0, 1, 0, 0, 1, 1, c3, -1], [0, 0, 0, 1, 0, 0, 1, c3, -1], [0, 0, 0, 0, 1, 0, -1, -c3 - 1, 1]])
is_fangiving(K_4_25,char_map_1 , [c3] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, -1, -1, d1 - 1, d1], [0, 0, 1, 0, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, 0, -1, -1], [0, 0, 0, 0, 1, 0, -1, d4 - 1, d4]])
is_fangiving(K_4_25,char_map_2 , [d1, d4] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, -1, b0, 0, 1], [0, 1, 0, 0, 0, -1, b1, 1, 2], [0, 0, 1, 0, 0, 1, -b1 - 1, -1, -1], [0, 0, 0, 1, 0, 0, b3, -1, -1], [0, 0, 0, 0, 1, 0, -1, 0, 1]])
is_fangiving(K_4_25,char_map_3 , [b0, b3, b1] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, 0, -1, -c2 - 1, -c2 - 1, 1], [0, 1, 0, 0, 0, -1, -c2 - 2, -c2 - 1, 1], [0, 0, 1, 0, 0, 1, c2 + 1, c2, -1], [0, 0, 0, 1, 0, 0, c3 + 1, c3, -1], [0, 0, 0, 0, 1, 0, -1, -1, 0]])
is_fangiving(K_4_25,char_map_4 , [c3, c2] )

## $K_4^{26}$

In [52]:
K_4_26=sc.PureSimplicialComplex(None, [[6, 9], [1, 2, 8], [1, 3, 8], [1, 6, 8], [2, 5, 7], [3, 4, 7], [3, 4, 8], [3, 4, 9], [1, 2, 5, 6], [4, 5, 7, 9]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 1, 0, 0, -1], [0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_4_26,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 2, 1, -1, -1], [0, 0, 0, 1, 0, 1, 0, -1, -1], [0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_4_26,char_map_1 , [] )

## $K_4^{27}$

In [54]:
K_4_27=sc.PureSimplicialComplex(None, [[1, 2, 6], [1, 4, 6], [1, 4, 9], [1, 6, 8], [2, 5, 7], [2, 5, 9], [3, 4, 8], [3, 4, 9], [3, 7, 8], [6, 7, 8], [3, 5, 7, 9]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, -1, 1, 0, 0], [0, 1, 0, 0, 0, 0, -1, 2, 1], [0, 0, 1, 0, 0, 0, 0, -1, -1], [0, 0, 0, 1, 0, -1, 1, -1, -1], [0, 0, 0, 0, 1, 0, -1, 1, 0]])
is_fangiving(K_4_27,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 0, -1], [0, 1, 0, 0, 0, -2, -1, 2, -1], [0, 0, 1, 0, 0, 1, 0, -1, 0], [0, 0, 0, 1, 0, 1, 1, -1, 0], [0, 0, 0, 0, 1, -1, -1, 1, -1]])
is_fangiving(K_4_27,char_map_1 , [] )

## $K_4^{28}$ -> some problems (no affecting the result)

In [56]:
K_4_28=sc.PureSimplicialComplex(None, [[1, 2, 7], [1, 2, 8], [1, 3, 7], [2, 5, 6], [2, 6, 8], [3, 4, 8], [3, 4, 9], [4, 6, 8], [5, 6, 9], [5, 7, 9]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 1, -1, -1, 0], [0, 1, 0, 0, 0, -1, 0, 1, 1], [0, 0, 1, 0, 0, 2, -1, -2, -1], [0, 0, 0, 1, 0, 1, -1, -2, -1], [0, 0, 0, 0, 1, -1, 0, 1, 0]])
is_fangiving(K_4_28,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, 0, -1, -1, -1, 1], [0, 0, 1, 0, 0, 2, 1, 0, -1], [0, 0, 0, 1, 0, 1, 1, 0, -1], [0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_4_28,char_map_1 , [] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, 0, -1, -1], [0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_4_28,char_map_2 , [] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, 0, -1, -1, -1, 0], [0, 0, 1, 0, 0, 0, -1, -2, -1], [0, 0, 0, 1, 0, 0, 0, -1, -1], [0, 0, 0, 0, 1, -1, -1, -1, -1]])
is_fangiving(K_4_28,char_map_3 , [] )

## $K_4^{29}$

In [57]:
K_4_29=sc.PureSimplicialComplex(None, [[2, 7], [6, 9], [1, 2, 6], [1, 6, 8], [3, 4, 9], [4, 7, 9], [1, 2, 5, 8], [1, 3, 5, 8], [3, 4, 5, 7], [3, 4, 5, 8]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 1, 0, 0, -1], [0, 0, 0, 0, 1, 0, -1, -1, 0]])
is_fangiving(K_4_29,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 2, 1, -1, -1], [0, 0, 0, 1, 0, 1, 0, -1, -1], [0, 0, 0, 0, 1, 1, 0, -1, 0]])
is_fangiving(K_4_29,char_map_1 , [] )

## $K_4^{30}$

In [59]:
K_4_30=sc.PureSimplicialComplex(None, [[2, 7], [1, 2, 6], [1, 2, 8], [1, 3, 8], [3, 4, 7], [3, 4, 8], [4, 7, 9], [5, 6, 9], [1, 5, 6, 8], [3, 4, 5, 9]] ,5)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 1, 0, 0, -1], [0, 0, 0, 0, 1, 0, 1, 0, -1]])
is_fangiving(K_4_30,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 2, 1, -1, -1], [0, 0, 0, 1, 0, 1, 0, -1, -1], [0, 0, 0, 0, 1, 0, 1, 0, -1]])
is_fangiving(K_4_30,char_map_1 , [] )

## $K_5^{31}$

In [ ]:
K_5_31=sc.PureSimplicialComplex(None, [[7, 9], [2, 5, 7], [3, 4, 9], [3, 6, 10], [5, 7, 10], [1, 2, 4, 8], [1, 2, 4, 9], [1, 2, 5, 8], [5, 6, 8, 10], [1, 3, 4, 6, 8]] ,6)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, -1, -1, 1], [0, 0, 1, 0, 0, 0, 2, 1, 0, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1, 0], [0, 0, 0, 0, 1, 0, -1, -1, 0, 0], [0, 0, 0, 0, 0, 1, 1, 0, 0, -1]])
is_fangiving(K_5_31,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1, 0], [0, 0, 0, 0, 1, 0, -1, -1, 0, 0], [0, 0, 0, 0, 0, 1, 1, 0, 0, -1]])
is_fangiving(K_5_31,char_map_1 , [] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, 0, -1, -1, 0], [0, 0, 0, 0, 1, 0, -1, 0, 1, 0], [0, 0, 0, 0, 0, 1, 1, 0, 0, -1]])
is_fangiving(K_5_31,char_map_2 , [] )

## $K_5^{32}$

In [63]:
K_4_32=sc.PureSimplicialComplex(None, [[8, 10], [1, 2, 8], [3, 4, 10], [3, 5, 9], [4, 7, 10], [1, 2, 6, 7], [1, 3, 5, 8], [2, 4, 6, 7], [4, 6, 7, 9], [1, 2, 5, 6, 9]] ,6)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, 0, 1], [0, 1, 0, 0, 0, 0, -1, 0, 1, 2], [0, 0, 1, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, -1, 0, 0, 1]])
is_fangiving(K_4_32,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, 0, 0, -1, -2, -1, 0], [0, 0, 1, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, -1, -1, -1, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, -1, -1, -1, 0]])
is_fangiving(K_4_32,char_map_1 , [] )

## $K_5^{33}$

In [64]:
a5 = sp.symbols('a5')
d5 = sp.symbols('d5')
K_4_33=sc.PureSimplicialComplex(None, [[2, 8], [8, 10], [1, 2, 7], [3, 6, 8], [4, 7, 10], [1, 2, 6, 9], [3, 4, 5, 10], [3, 5, 6, 9], [1, 4, 5, 7, 9]] ,6)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, -1, d0], [0, 1, 0, 0, 0, 0, -1, -2, -1, d1], [0, 0, 1, 0, 0, 0, 1, 1, 0, -1], [0, 0, 0, 1, 0, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 0, 1, 2, 0, -1], [0, 0, 0, 0, 0, 1, 0, -1, -1, d5]])
is_fangiving(K_4_33,char_map_0 , [d1, d5, d0] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, d0 - 1, d0 - 1, d0], [0, 1, 0, 0, 0, 0, -1, d1 - 2, d1 - 1, d1], [0, 0, 1, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 0, 0, 0, -1, -1], [0, 0, 0, 0, 0, 1, 0, -1, -1, 0]])
is_fangiving(K_4_33,char_map_1 , [d1, d0] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -c2 - 1, -c2 - 1, 1], [0, 1, 0, 0, 0, 0, -1, -c2 - 2, -c2 - 1, 1], [0, 0, 1, 0, 0, 0, 1, c2 + 1, c2, -1], [0, 0, 0, 1, 0, 0, 0, c3 + 1, c3, -1], [0, 0, 0, 0, 1, 0, 1, c2 + 2, c2, -1], [0, 0, 0, 0, 0, 1, 0, -1, -1, 0]])
is_fangiving(K_4_33,char_map_2 , [c3, c2] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, -2, -1, 0], [0, 0, 1, 0, 0, 0, c2 + 1, 2*c2 + 1, c2, -1], [0, 0, 0, 1, 0, 0, c3, 2*c3 + 1, c3, -1], [0, 0, 0, 0, 1, 0, c4 + 1, 2*c4 + 2, c4, -1], [0, 0, 0, 0, 0, 1, 0, -1, -1, 0]])
is_fangiving(K_4_33,char_map_3 , [c3, c2, c4] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, a2, a2 - 1, -1, -1], [0, 0, 0, 1, 0, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 0, a4, a4, -1, -1], [0, 0, 0, 0, 0, 1, a5, a5 - 1, -1, 0]])
is_fangiving(K_4_33,char_map_4 , [a4, a5, a2] )

## $K_5^{34}$

In [65]:
K_4_34=sc.PureSimplicialComplex(None, [[8, 10], [1, 2, 7], [1, 2, 8], [2, 4, 7], [4, 7, 10], [1, 2, 5, 9], [1, 3, 5, 8], [3, 4, 6, 10], [3, 5, 6, 9], [4, 6, 7, 9]] ,6)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, 0, 1], [0, 1, 0, 0, 0, 0, -1, 0, 1, 2], [0, 0, 1, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_4_34,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, 0, 0, -1, -2, -1, 0], [0, 0, 1, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, -1, -1, -1, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_4_34,char_map_1 , [] )

## $K_5^{35}$

In [67]:
c5 = sp.symbols('c5')

K_4_35=sc.PureSimplicialComplex(None, [[2, 5, 8], [3, 4, 9], [3, 4, 10], [3, 5, 8], [3, 5, 9], [3, 8, 10], [6, 8, 10], [1, 2, 5, 9], [1, 2, 6, 7], [1, 4, 7, 9], [4, 6, 7, 10]] ,6)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, -1, d0], [0, 1, 0, 0, 0, 0, -1, -2, -1, d1], [0, 0, 1, 0, 0, 0, 1, 1, 0, -1], [0, 0, 0, 1, 0, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1, d4], [0, 0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_4_35,char_map_0 , [d0, d4, d1] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, -c3 - 1, 1], [0, 1, 0, 0, 0, 0, -1, -2, -2*c3 - 1, 2], [0, 0, 1, 0, 0, 0, 1, 1, c3, -1], [0, 0, 0, 1, 0, 0, 0, 1, c3, -1], [0, 0, 0, 0, 1, 0, 0, -1, -c3 - 1, 1], [0, 0, 0, 0, 0, 1, -1, -1, c5, 0]])
is_fangiving(K_4_35,char_map_1 , [c5, c3] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, 0, -1, -1, d1 - 1, d1], [0, 0, 1, 0, 0, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, 0, 0, -1, -1], [0, 0, 0, 0, 1, 0, 0, -1, d4 - 1, d4], [0, 0, 0, 0, 0, 1, -1, 0, 1, 0]])
is_fangiving(K_4_35,char_map_2 , [d4, d1] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, b0, 0, 1], [0, 1, 0, 0, 0, 0, -1, b1, 1, 2], [0, 0, 1, 0, 0, 0, 1, -b1 - 1, -1, -1], [0, 0, 0, 1, 0, 0, 0, b3, -1, -1], [0, 0, 0, 0, 1, 0, 0, -1, 0, 1], [0, 0, 0, 0, 0, 1, -1, b1 + 1, 1, 0]])
is_fangiving(K_4_35,char_map_3 , [b1, b3, b0] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -c2 - 1, -c2 - 1, 1], [0, 1, 0, 0, 0, 0, -1, -c2 - 2, -c2 - 1, 1], [0, 0, 1, 0, 0, 0, 1, c2 + 1, c2, -1], [0, 0, 0, 1, 0, 0, 0, c3 + 1, c3, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, -1, c5 - 1, c5, 0]])
is_fangiving(K_4_35,char_map_4 , [c5, c3, c2] )

## $K_5^{36}$

In [69]:
K_4_36=sc.PureSimplicialComplex(None, [[1, 7, 9], [2, 5, 8], [3, 4, 9], [3, 4, 10], [3, 8, 9], [7, 8, 9], [1, 2, 6, 7], [1, 4, 6, 7], [1, 4, 6, 10], [2, 5, 6, 10], [3, 5, 8, 10]] ,6)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, 1, 0, 0], [0, 1, 0, 0, 0, 0, 0, -1, 2, 1], [0, 0, 1, 0, 0, 0, 0, 0, -1, -1], [0, 0, 0, 1, 0, 0, -1, 1, -1, -1], [0, 0, 0, 0, 1, 0, 0, -1, 1, 0], [0, 0, 0, 0, 0, 1, -1, 0, 1, 0]])
is_fangiving(K_4_36,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, 0, 0, -1], [0, 1, 0, 0, 0, 0, -2, -1, 2, -1], [0, 0, 1, 0, 0, 0, 1, 0, -1, 0], [0, 0, 0, 1, 0, 0, 1, 1, -1, 0], [0, 0, 0, 0, 1, 0, -1, -1, 1, -1], [0, 0, 0, 0, 0, 1, -1, 0, 1, -1]])
is_fangiving(K_4_36,char_map_1 , [] )

## $K_5^{37}$

In [77]:
K_4_37=sc.PureSimplicialComplex(None, [[1, 2, 8], [1, 2, 9], [2, 5, 7], [2, 7, 9], [3, 4, 9], [4, 7, 9], [5, 7, 10], [1, 3, 6, 8], [3, 4, 6, 10], [5, 6, 8, 10]] ,6)
char_map_0 =sp.Matrix([[1, 0, 0, 0, 0, 0, 1, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, 0, 1, 1], [0, 0, 1, 0, 0, 0, 2, -1, -2, -1], [0, 0, 0, 1, 0, 0, 1, -1, -2, -1], [0, 0, 0, 0, 1, 0, -1, 0, 1, 0], [0, 0, 0, 0, 0, 1, 1, -1, -1, -1]])
is_fangiving(K_4_37,char_map_0 , [] )
char_map_1 =sp.Matrix([[1, 0, 0, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, -1, -1, 1], [0, 0, 1, 0, 0, 0, 2, 1, 0, -1], [0, 0, 0, 1, 0, 0, 1, 1, 0, -1], [0, 0, 0, 0, 1, 0, -1, -1, 0, 0], [0, 0, 0, 0, 0, 1, 1, 0, 0, -1]])
is_fangiving(K_4_37,char_map_1 , [] )
char_map_2 =sp.Matrix([[1, 0, 0, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, 0, 0, -1, -1], [0, 0, 0, 0, 1, 0, -1, -1, 0, 0], [0, 0, 0, 0, 0, 1, 1, 0, 0, -1]])
is_fangiving(K_4_37,char_map_2 , [] )
char_map_3 =sp.Matrix([[1, 0, 0, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, -1, -1, 0], [0, 0, 1, 0, 0, 0, 0, -1, -2, -1], [0, 0, 0, 1, 0, 0, 0, 0, -1, -1], [0, 0, 0, 0, 1, 0, -1, -1, -1, -1], [0, 0, 0, 0, 0, 1, 0, -1, -1, -1]])
is_fangiving(K_4_37,char_map_3 , [] )

## $K_5^{38}$

In [ ]:
K_4_38=sc.PureSimplicialComplex(None, [[7, 10], [1, 2, 7], [1, 7, 9], [2, 6, 8], [3, 4, 10], [4, 8, 10], [1, 3, 5, 9], [3, 4, 5, 9], [1, 2, 5, 6, 9], [3, 4, 5, 6, 8]] ,6)
char_map_0 =sp.Matrix([[1, 0, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, 0, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, 0, -1, 0, 1]])
is_fangiving(K_4_38,char_map_0 , [] )
char_map_1 =sp.Matrix([[1, 0, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 2, 1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1, -1], [0, 0, 0, 0, 1, 0, 1, 0, -1, 0], [0, 0, 0, 0, 0, 1, 0, -1, 0, 1]])
is_fangiving(K_4_38,char_map_1 , [] )

## $K_5^{39}$

In [ ]:
K_4_39=sc.PureSimplicialComplex(None, [[2, 5, 10], [2, 6, 8], [3, 4, 9], [3, 4, 10], [3, 8, 10], [1, 2, 5, 7], [1, 4, 5, 7], [1, 4, 5, 10], [1, 4, 7, 9], [1, 6, 7, 9], [3, 6, 8, 9], [6, 7, 8, 9]] ,6)
char_map_0 =sp.Matrix([[1, 0, 0, 0, 0, 0, -1, 0, 0, -1], [0, 1, 0, 0, 0, 0, -2, -1, 2, -1], [0, 0, 1, 0, 0, 0, 1, 0, -1, 0], [0, 0, 0, 1, 0, 0, 1, 1, -1, 0], [0, 0, 0, 0, 1, 0, -1, 0, 1, -1], [0, 0, 0, 0, 0, 1, -1, -1, 0, -1]])
is_fangiving(K_4_39,char_map_0 , [] )

## $K_5^{40}$

In [78]:
K_4_40=sc.PureSimplicialComplex(None, [[7, 10], [1, 7, 9], [2, 6, 8], [3, 4, 10], [1, 2, 5, 9], [1, 2, 6, 7], [1, 3, 5, 9], [3, 4, 5, 8], [3, 4, 5, 9], [4, 6, 8, 10]] ,6)
char_map_0 =sp.Matrix([[1, 0, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, 0, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_4_40,char_map_0 , [] )
char_map_1 =sp.Matrix([[1, 0, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 2, 1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1, -1], [0, 0, 0, 0, 1, 0, 1, 0, -1, 0], [0, 0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_4_40,char_map_1 , [] )

## $K_5^{41}$

In [79]:
K_4_41=sc.PureSimplicialComplex(None, [[1, 2, 8], [1, 2, 9], [1, 3, 8], [2, 5, 7], [2, 7, 9], [5, 8, 10], [1, 3, 4, 9], [3, 4, 6, 9], [3, 4, 6, 10], [4, 6, 7, 9], [5, 6, 7, 10]] ,6)
char_map_0 =sp.Matrix([[1, 0, 0, 0, 0, 0, 1, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, 0, 1, 1], [0, 0, 1, 0, 0, 0, 2, -1, -2, -1], [0, 0, 0, 1, 0, 0, 1, -1, -2, -1], [0, 0, 0, 0, 1, 0, -1, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_4_41,char_map_0 , [] )
char_map_1 =sp.Matrix([[1, 0, 0, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, 0, 0, -1, -1], [0, 0, 0, 0, 1, 0, -1, -1, 0, 0], [0, 0, 0, 0, 0, 1, 0, 1, 0, -1]])
is_fangiving(K_4_41,char_map_1 , [] )

## $K_5^{42}$

In [ ]:
K_5_42=sc.PureSimplicialComplex(None, [[2, 8], [1, 2, 7], [4, 8, 10], [6, 7, 10], [1, 2, 5, 9], [1, 3, 5, 9], [1, 6, 7, 9], [3, 4, 5, 8], [3, 4, 5, 9], [3, 4, 6, 10]] ,6)
char_map_0 =sp.Matrix([[1, 0, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, 0, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, 0, 1, 0, -1]])
is_fangiving(K_5_42,char_map_0 , [] )
char_map_1 =sp.Matrix([[1, 0, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 2, 1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1, -1], [0, 0, 0, 0, 1, 0, 1, 0, -1, 0], [0, 0, 0, 0, 0, 1, 0, 1, 0, -1]])
is_fangiving(K_5_42,char_map_1 , [] )

## $K_5^{43}$

In [ ]:
K_5_43=sc.PureSimplicialComplex(None, [[2, 8], [1, 7, 9], [3, 4, 9], [3, 8, 9], [3, 8, 10], [7, 8, 9], [1, 2, 5, 7], [2, 5, 6, 10], [3, 4, 6, 10], [1, 4, 5, 6, 7], [1, 4, 5, 6, 10]] ,6)
char_map_0 =sp.Matrix([[1, 0, 0, 0, 0, 0, -1, 1, 0, 0], [0, 1, 0, 0, 0, 0, 0, -1, 2, 1], [0, 0, 1, 0, 0, 0, 0, 0, -1, -1], [0, 0, 0, 1, 0, 0, -1, 1, -1, -1], [0, 0, 0, 0, 1, 0, -1, 0, 1, 0], [0, 0, 0, 0, 0, 1, -1, 0, 0, -1]])
is_fangiving(K_5_43,char_map_0 , [] )

## $K_5^{44}$

In [82]:
K_5_44=sc.PureSimplicialComplex(None, [[2, 5, 10], [2, 6, 8], [3, 4, 9], [3, 4, 10], [3, 8, 9], [3, 8, 10], [1, 4, 5, 7], [1, 4, 5, 10], [1, 4, 7, 9], [1, 6, 7, 9], [6, 7, 8, 9], [1, 2, 5, 6, 7]] ,6)
char_map_0 =sp.Matrix([[1, 0, 0, 0, 0, 0, -1, 1, 0, 0], [0, 1, 0, 0, 0, 0, 0, -1, 2, 1], [0, 0, 1, 0, 0, 0, 0, 0, -1, -1], [0, 0, 0, 1, 0, 0, -1, 1, -1, -1], [0, 0, 0, 0, 1, 0, -1, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, -1, 1, 1]])
is_fangiving(K_5_44,char_map_0 , [] )
char_map_1 =sp.Matrix([[1, 0, 0, 0, 0, 0, -1, 0, 0, -1], [0, 1, 0, 0, 0, 0, -2, -1, 2, -1], [0, 0, 1, 0, 0, 0, 1, 0, -1, 0], [0, 0, 0, 1, 0, 0, 1, 1, -1, 0], [0, 0, 0, 0, 1, 0, -1, 0, 1, -1], [0, 0, 0, 0, 0, 1, -2, -1, 1, -1]])
is_fangiving(K_5_44,char_map_1 , [] )

## $K_5^{45}$

In [83]:
K_5_45=sc.PureSimplicialComplex(None, [[1, 7, 9], [2, 5, 8], [2, 6, 8], [3, 4, 10], [6, 7, 10], [1, 2, 5, 9], [1, 2, 6, 7], [1, 3, 5, 9], [3, 4, 5, 8], [3, 4, 5, 9], [4, 6, 8, 10]] ,6)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, 0, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_5_45,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 2, 1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1, -1], [0, 0, 0, 0, 1, 0, 1, 0, -1, 0], [0, 0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_5_45,char_map_1 , [] )

## $K_5^{46}$

In [84]:
K_5_46=sc.PureSimplicialComplex(None, [[1, 2, 7], [1, 4, 7], [1, 4, 10], [2, 5, 10], [3, 4, 9], [3, 4, 10], [1, 6, 7, 9], [2, 5, 6, 8], [3, 5, 8, 10], [3, 6, 8, 9], [6, 7, 8, 9]] ,6)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, 0, 0, -1], [0, 1, 0, 0, 0, 0, -2, -1, 2, -1], [0, 0, 1, 0, 0, 0, 1, 0, -1, 0], [0, 0, 0, 1, 0, 0, 1, 1, -1, 0], [0, 0, 0, 0, 1, 0, -1, -1, 1, -1], [0, 0, 0, 0, 0, 1, -1, -1, 0, -1]])
is_fangiving(K_5_46,char_map_0 , [] )

## $K_5^{47}$

In [85]:
K_5_47=sc.PureSimplicialComplex(None, [[2, 5, 7], [3, 6, 10], [3, 8, 10], [5, 8, 10], [6, 7, 9], [1, 2, 4, 8], [1, 2, 4, 9], [1, 2, 5, 8], [1, 3, 4, 8], [1, 3, 4, 9], [3, 4, 6, 9], [5, 6, 7, 10]] ,6)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1, 0], [0, 0, 0, 0, 1, 0, -1, -1, 0, 0], [0, 0, 0, 0, 0, 1, 0, 1, 0, -1]])
is_fangiving(K_5_47,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, 0, -1, -1, 0], [0, 0, 0, 0, 1, 0, -1, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_5_47,char_map_1 , [] )

## $K_5^{48}$ -> problem

In [89]:
K_5_48=sc.PureSimplicialComplex(None, [[2, 8], [7, 10], [1, 2, 7], [4, 8, 10], [1, 2, 5, 9], [1, 6, 7, 9], [3, 4, 5, 8], [3, 4, 6, 10], [1, 3, 5, 6, 9], [3, 4, 5, 6, 9]] ,6)
print("first:")
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, 0, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_5_48,char_map_0 , [] )
print("second")
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 2, 1, -1, -1], [0, 0, 0, 1, 0, 0, 1, 0, -1, -1], [0, 0, 0, 0, 1, 0, 1, 0, -1, 0], [0, 0, 0, 0, 0, 1, 1, 1, -1, -1]])
is_fangiving(K_5_48,char_map_1 , [] )

first:
second


## $K_6^{49}$

In [91]:
K_5_49=sc.PureSimplicialComplex(None, [[3, 4, 10], [3, 4, 11], [3, 9, 10], [1, 4, 6, 8], [1, 4, 6, 11], [1, 4, 8, 10], [1, 7, 8, 10], [2, 5, 6, 11], [2, 5, 7, 9], [3, 5, 9, 11], [7, 8, 9, 10], [1, 2, 6, 7, 8]] ,7)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, 1, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, -1, 2, 1], [0, 0, 1, 0, 0, 0, 0, 0, 0, -1, -1], [0, 0, 0, 1, 0, 0, 0, -1, 1, -1, -1], [0, 0, 0, 0, 1, 0, 0, 0, -1, 1, 0], [0, 0, 0, 0, 0, 1, 0, -1, 0, 1, 0], [0, 0, 0, 0, 0, 0, 1, 0, -1, 1, 1]])
is_fangiving(K_5_49,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, 0, 0, -1], [0, 1, 0, 0, 0, 0, 0, -2, -1, 2, -1], [0, 0, 1, 0, 0, 0, 0, 1, 0, -1, 0], [0, 0, 0, 1, 0, 0, 0, 1, 1, -1, 0], [0, 0, 0, 0, 1, 0, 0, -1, -1, 1, -1], [0, 0, 0, 0, 0, 1, 0, -1, 0, 1, -1], [0, 0, 0, 0, 0, 0, 1, -2, -1, 1, -1]])
is_fangiving(K_5_49,char_map_1 , [] )

## $K_6^{50}$

In [92]:
K_5_50=sc.PureSimplicialComplex(None, [[9, 11], [1, 2, 9], [4, 8, 11], [1, 2, 6, 8], [1, 3, 5, 9], [2, 4, 6, 8], [3, 4, 7, 11], [3, 5, 7, 10], [1, 2, 5, 6, 10], [4, 6, 7, 8, 10]] ,7)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, -1, 0, 1], [0, 1, 0, 0, 0, 0, 0, -1, 0, 1, 2], [0, 0, 1, 0, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 0, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, 0, -1, 0, 0, 1], [0, 0, 0, 0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_5_50,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, 0, 0, 0, -1, -2, -1, 0], [0, 0, 1, 0, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 0, -1, -1, -1, -1], [0, 0, 0, 0, 1, 0, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, 0, -1, -1, -1, 0], [0, 0, 0, 0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_5_50,char_map_1 , [] )

## $K_6^{51}$

In [93]:
K_5_51=sc.PureSimplicialComplex(None, [[8, 11], [1, 2, 8], [2, 6, 9], [4, 9, 11], [1, 7, 8, 10], [3, 4, 7, 11], [1, 2, 5, 6, 10], [1, 3, 5, 7, 10], [3, 4, 5, 6, 9], [3, 4, 5, 7, 10]] ,7)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, -1], [0, 0, 0, 0, 1, 0, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, 0, 0, -1, 0, 1], [0, 0, 0, 0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_5_51,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 0, 2, 1, -1, -1], [0, 0, 0, 1, 0, 0, 0, 1, 0, -1, -1], [0, 0, 0, 0, 1, 0, 0, 1, 0, -1, 0], [0, 0, 0, 0, 0, 1, 0, 0, -1, 0, 1], [0, 0, 0, 0, 0, 0, 1, 1, 1, -1, -1]])
is_fangiving(K_5_51,char_map_1 , [] )

## $K_6^{52}$

In [94]:
K_5_52=sc.PureSimplicialComplex(None, [[1, 2, 10], [2, 8, 10], [1, 2, 7, 9], [1, 3, 4, 9], [1, 3, 4, 10], [1, 3, 7, 9], [2, 5, 7, 8], [3, 4, 6, 10], [3, 4, 6, 11], [4, 6, 8, 10], [5, 6, 8, 11], [5, 7, 9, 11]] ,7)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, 1, -1, -1, 0], [0, 1, 0, 0, 0, 0, 0, -1, 0, 1, 1], [0, 0, 1, 0, 0, 0, 0, 2, -1, -2, -1], [0, 0, 0, 1, 0, 0, 0, 1, -1, -2, -1], [0, 0, 0, 0, 1, 0, 0, -1, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, -1, -1], [0, 0, 0, 0, 0, 0, 1, -1, 0, 2, 1]])
is_fangiving(K_5_52,char_map_0 , [] )

## $K_6^{53}$

In [95]:
a5=sp.symbols('a5')
c6=sp.symbols('c6')
d6=sp.symbols('d6')
K_5_53=sc.PureSimplicialComplex(None, [[2, 6, 9], [2, 7, 9], [3, 6, 9], [3, 9, 11], [7, 9, 11], [1, 2, 6, 10], [1, 2, 7, 8], [3, 4, 5, 11], [3, 5, 6, 10], [4, 7, 8, 11], [1, 4, 5, 8, 10]] ,7)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, -1, -1, d0], [0, 1, 0, 0, 0, 0, 0, -1, -2, -1, d1], [0, 0, 1, 0, 0, 0, 0, 1, 1, 0, -1], [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 0, 0, 1, 2, 0, -1], [0, 0, 0, 0, 0, 1, 0, 0, -1, -1, d5], [0, 0, 0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_5_53,char_map_0 , [d1, d0, d5] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, d0 - 1, d0 - 1, d0], [0, 1, 0, 0, 0, 0, 0, -1, d1 - 2, d1 - 1, d1], [0, 0, 1, 0, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 0, 0, 0, 0, -1, -1], [0, 0, 0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 0, 1, -1, d6 - 1, d6, d6]])
is_fangiving(K_5_53,char_map_1 , [d1, d6, d0] )
char_map_2 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, -c2 - 1, -c2 - 1, 1], [0, 1, 0, 0, 0, 0, 0, -1, -c2 - 2, -c2 - 1, 1], [0, 0, 1, 0, 0, 0, 0, 1, c2 + 1, c2, -1], [0, 0, 0, 1, 0, 0, 0, 0, c3 + 1, c3, -1], [0, 0, 0, 0, 1, 0, 0, 1, c2 + 2, c2, -1], [0, 0, 0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 0, 1, -1, c6 - 1, c6, 0]])
is_fangiving(K_5_53,char_map_2 , [c2, c3, c6] )
char_map_3 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, 0, 0, -1, -2, -1, 0], [0, 0, 1, 0, 0, 0, 0, c2 + 1, 2*c2 + 1, c2, -1], [0, 0, 0, 1, 0, 0, 0, c3, 2*c3 + 1, c3, -1], [0, 0, 0, 0, 1, 0, 0, c4 + 1, 2*c4 + 2, c4, -1], [0, 0, 0, 0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_5_53,char_map_3 , [c4, c2, c3] )
char_map_4 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, 0, 0, 1], [0, 1, 0, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 0, a2, a2 - 1, -1, -1], [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, -1], [0, 0, 0, 0, 1, 0, 0, a4, a4, -1, -1], [0, 0, 0, 0, 0, 1, 0, a5, a5 - 1, -1, 0], [0, 0, 0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_5_53,char_map_4 , [a2, a5, a4] )

## $K_6^{54}$

In [96]:
K_5_54=sc.PureSimplicialComplex(None, [[1, 2, 9], [1, 2, 10], [2, 5, 8], [2, 8, 10], [1, 3, 4, 10], [1, 3, 7, 9], [3, 4, 6, 10], [4, 6, 8, 10], [5, 6, 8, 11], [5, 7, 9, 11], [3, 4, 6, 7, 11]] ,7)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, 1, -1, -1, 0], [0, 1, 0, 0, 0, 0, 0, -1, 0, 1, 1], [0, 0, 1, 0, 0, 0, 0, 2, -1, -2, -1], [0, 0, 0, 1, 0, 0, 0, 1, -1, -2, -1], [0, 0, 0, 0, 1, 0, 0, -1, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, -1, -1], [0, 0, 0, 0, 0, 0, 1, 1, -1, -1, -1]])
is_fangiving(K_5_54,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 0, 1, 0, -1, -1], [0, 0, 0, 1, 0, 0, 0, 0, 0, -1, -1], [0, 0, 0, 0, 1, 0, 0, -1, -1, 0, 0], [0, 0, 0, 0, 0, 1, 0, 0, 1, 0, -1], [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, -1]])
is_fangiving(K_5_54,char_map_1 , [] )

## $K_6^{55}$

In [98]:
K_5_55=sc.PureSimplicialComplex(None, [[3, 4, 11], [3, 5, 10], [3, 9, 11], [4, 8, 11], [7, 9, 11], [1, 2, 7, 9], [1, 3, 5, 9], [4, 6, 8, 10], [1, 2, 5, 6, 10], [1, 2, 6, 7, 8], [2, 4, 6, 7, 8]] ,7)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, 0, -1, 0, 1], [0, 1, 0, 0, 0, 0, 0, -1, -2, -1, 0], [0, 0, 1, 0, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 0, -1, -1, -1, -1], [0, 0, 0, 0, 1, 0, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, 0, -1, -1, -1, 0], [0, 0, 0, 0, 0, 0, 1, -1, -1, 0, 0]])
is_fangiving(K_5_55,char_map_0 , [] )

## $K_6^{56}$

In [100]:
K_5_56=sc.PureSimplicialComplex(None, [[2, 5, 11], [3, 4, 10], [3, 4, 11], [3, 9, 11], [1, 4, 5, 8], [1, 4, 5, 11], [1, 4, 8, 10], [2, 6, 7, 9], [3, 6, 9, 10], [1, 2, 5, 7, 8], [1, 6, 7, 8, 10], [6, 7, 8, 9, 10]] ,7)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, 0, 0, -1], [0, 1, 0, 0, 0, 0, 0, -2, -1, 2, -1], [0, 0, 1, 0, 0, 0, 0, 1, 0, -1, 0], [0, 0, 0, 1, 0, 0, 0, 1, 1, -1, 0], [0, 0, 0, 0, 1, 0, 0, -1, 0, 1, -1], [0, 0, 0, 0, 0, 1, 0, -1, -1, 0, -1], [0, 0, 0, 0, 0, 0, 1, -2, -1, 1, -1]])
is_fangiving(K_5_56,char_map_0 , [] )

## $K_6^{57}$

In [102]:
K_5_57=sc.PureSimplicialComplex(None, [[2, 6, 9], [6, 8, 11], [7, 8, 11], [1, 2, 5, 10], [1, 2, 6, 8], [1, 2, 8, 10], [1, 7, 8, 10], [3, 4, 5, 9], [3, 4, 7, 11], [4, 6, 9, 11], [1, 3, 5, 7, 10], [3, 4, 5, 7, 10]] ,7)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, -1, -1, -1, 0], [0, 1, 0, 0, 0, 0, 0, -1, -1, 0, 1], [0, 0, 1, 0, 0, 0, 0, 0, -1, -1, -1], [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, -1], [0, 0, 0, 0, 1, 0, 0, 0, -1, -1, 0], [0, 0, 0, 0, 0, 1, 0, -1, -1, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0, 0, -1, -1]])
is_fangiving(K_5_57,char_map_0 , [] )

## $K_7^{58}$

In [105]:
K_7_58=sc.PureSimplicialComplex(None, [[3, 4, 11], [3, 4, 12], [1, 4, 5, 9], [1, 4, 5, 12], [1, 4, 9, 11], [2, 5, 8, 12], [3, 6, 10, 11], [3, 8, 10, 12], [1, 2, 5, 7, 9], [1, 6, 7, 9, 11], [2, 6, 7, 8, 10], [6, 7, 9, 10, 11]] ,8)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, -1], [0, 1, 0, 0, 0, 0, 0, 0, -2, -1, 2, -1], [0, 0, 1, 0, 0, 0, 0, 0, 1, 0, -1, 0], [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, -1, 0], [0, 0, 0, 0, 1, 0, 0, 0, -1, 0, 1, -1], [0, 0, 0, 0, 0, 1, 0, 0, -1, -1, 0, -1], [0, 0, 0, 0, 0, 0, 1, 0, -2, -1, 1, -1], [0, 0, 0, 0, 0, 0, 0, 1, -1, -1, 1, -1]])
is_fangiving(K_7_58,char_map_0 , [] )

## $L_3^1$

In [ ]:
L_3_1=sc.PureSimplicialComplex(None, [[2, 5], [5, 7], [5, 8], [1, 2, 6], [1, 2, 7], [1, 6, 8], [2, 3, 7], [3, 4, 7], [3, 4, 8], [4, 6, 8], [1, 3, 4, 6]] ,4)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 1], [0, 1, 0, 0, -1, 1, -1, 0], [0, 0, 1, 0, -2, 1, 0, -1], [0, 0, 0, 1, -1, 0, 1, -1]])
is_fangiving(L_3_1,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 0, -1, -1, 0], [0, 1, 0, 0, -1, 1, 0, -1], [0, 0, 1, 0, -3, 2, 1, -1], [0, 0, 0, 1, -2, 1, 1, -1]])
is_fangiving(L_3_1,char_map_1 , [] )

## $L_3^2$

In [107]:
L_3_2=sc.PureSimplicialComplex(None, [[1, 8], [1, 2, 5], [1, 2, 6], [1, 5, 7], [2, 3, 6], [2, 4, 5], [2, 4, 6], [3, 4, 6], [3, 4, 7], [3, 6, 8], [3, 7, 8], [4, 5, 7], [5, 7, 8]] ,4)
char_map_0 =sp. Matrix([[1, 0, 0, 0, 3, 2, -2, -1], [0, 1, 0, 0, -1, -1, 0, 0], [0, 0, 1, 0, 1, 0, -1, 0], [0, 0, 0, 1, 2, 1, -1, -1]])
is_fangiving(L_3_2,char_map_0 , [] )
char_map_1 =sp. Matrix([[1, 0, 0, 0, 2, -2, -3, -1], [0, 1, 0, 0, -1, 0, 1, 0], [0, 0, 1, 0, 0, -1, -1, 0], [0, 0, 0, 1, 1, -1, -1, -1]])
is_fangiving(L_3_2,char_map_1 , [] )

Matrix([[1]])

Matrix([[0]])

Matrix([[1]])

Matrix([[0]])

Matrix([[1]])

...

Matrix([[0]])

Matrix([
[1],
[2],
[1],
[1],
[1],
[3],
[3]])

Matrix([
[0],
[0],
[0],
[0],
[0],
[1],
[1]])

Matrix([[1]])

Matrix([[0]])